In [ ]:
# Stable Diffusion 1.4
# IMPORTANT: Restart the kernel before running this. The old kernel still had
# transformers 5.9.0 loaded in memory, which diffusers 0.38.0 cannot use (so it
# reported "transformers not found"). The env now has transformers 4.57.6, which
# works once a fresh kernel loads it from disk.
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float32,
)
pipe = pipe.to("mps")
pipe.enable_attention_slicing()  # recommended on MPS to avoid memory spikes

In [1]:
import torch

In [36]:
import transformers
import diffusers

In [37]:
import transformers, diffusers
print(transformers.__version__)
print(diffusers.__version__)

5.9.0
0.38.0


In [ ]:
import transformers
from diffusers import StableDiffusionPipeline
import torch



ImportError: 
StableDiffusionPipeline requires the transformers library but it was not found in your environment. You can install it with pip: `pip
install transformers`


In [39]:
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float32
)
pipe = pipe.to("mps")

# Recommended on MPS: avoids memory spikes
pipe.enable_attention_slicing()

image = pipe("a photo of an astronaut riding a horse").images[0]
image.save("output.png")

ImportError: 
StableDiffusionPipeline requires the transformers library but it was not found in your environment. You can install it with pip: `pip
install transformers`


In [13]:
print(dict(model['ema'].unet.out_conv.named_modules()))



{'': MPConv()}


In [18]:
import sys
sys.path.insert(0, "./edm2")
import pickle
import torch
import numpy as np
from PIL import Image

# Load
with open("edm2-s-64.pkl", "rb") as f:
    data = pickle.load(f)

model = data['ema']
model.eval()
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = model.to(device)

# Karras sigma schedule
def get_sigmas(n=32, sigma_max=80, sigma_min=0.002, rho=7):
    steps = torch.arange(n, device=device) / (n - 1)
    sigmas = (sigma_max ** (1/rho) + steps * (sigma_min ** (1/rho) - sigma_max ** (1/rho))) ** rho
    return torch.cat([sigmas, torch.zeros(1, device=device)])

# Euler denoising loop
with torch.no_grad():
    sigmas = get_sigmas()
    x = torch.randn(1, 3, 64, 64, device=device) * sigmas[0]
    for i in range(len(sigmas) - 1):
        denoised = model(x, sigmas[i].expand(1))
        d = (x - denoised) / sigmas[i]
        x = x + d * (sigmas[i+1] - sigmas[i])

# Save
img = x[0].clamp(-1, 1).cpu().numpy()
img = ((img + 1) / 2 * 255).astype(np.uint8).transpose(1, 2, 0)
Image.fromarray(img).save("sample.png")
print("saved sample.png")

UnpicklingError: invalid load key, '<'.

In [19]:
import sys
sys.path.insert(0, "./edm2")
import requests
import pickle
import torch
import numpy as np
from PIL import Image

# Download
url = "https://nvlabs-fi-cdn.nvidia.com/edm2/posthoc-reconstructions/edm2-img64-s-2147483-0.015.pkl"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, stream=True)
with open("edm2-s-64.pkl", "wb") as f:
    for chunk in r.iter_content(chunk_size=8192):
        f.write(chunk)

# Load
with open("edm2-s-64.pkl", "rb") as f:
    data = pickle.load(f)

model = data['ema']
model.eval()
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = model.to(device)

# Karras sigma schedule
def get_sigmas(n=32, sigma_max=80, sigma_min=0.002, rho=7):
    steps = torch.arange(n, device=device) / (n - 1)
    sigmas = (sigma_max ** (1/rho) + steps * (sigma_min ** (1/rho) - sigma_max ** (1/rho))) ** rho
    return torch.cat([sigmas, torch.zeros(1, device=device)])

# Sample
with torch.no_grad():
    sigmas = get_sigmas()
    x = torch.randn(1, 3, 64, 64, device=device) * sigmas[0]
    for i in range(len(sigmas) - 1):
        denoised = model(x, sigmas[i].expand(1))
        d = (x - denoised) / sigmas[i]
        x = x + d * (sigmas[i+1] - sigmas[i])

# Save
img = x[0].clamp(-1, 1).cpu().numpy()
img = ((img + 1) / 2 * 255).astype(np.uint8).transpose(1, 2, 0)
Image.fromarray(img).save("sample.png")
print("saved sample.png")

UnpicklingError: invalid load key, '<'.

In [21]:
import sys
sys.path.insert(0, "./edm2")
import pickle
import torch
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download

# Download
path = hf_hub_download(
    repo_id="nvidia/DirectDiscriminativeOptimization",
    filename="edm2-img64-s-ddo.pkl"
)

# Load
with open(path, "rb") as f:
    data = pickle.load(f)

model = data['ema']
model.eval()
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = model.to(device)

# Karras sigma schedule
def get_sigmas(n=32, sigma_max=80, sigma_min=0.002, rho=7):
    steps = torch.arange(n, device=device) / (n - 1)
    sigmas = (sigma_max ** (1/rho) + steps * (sigma_min ** (1/rho) - sigma_max ** (1/rho))) ** rho
    return torch.cat([sigmas, torch.zeros(1, device=device)])

# Sample
with torch.no_grad():
    sigmas = get_sigmas()
    x = torch.randn(1, 3, 64, 64, device=device) * sigmas[0]
    for i in range(len(sigmas) - 1):
        denoised = model(x, sigmas[i].expand(1))
        d = (x - denoised) / sigmas[i]
        x = x + d * (sigmas[i+1] - sigmas[i])

# Save
img = x[0].clamp(-1, 1).cpu().numpy()
img = ((img + 1) / 2 * 255).astype(np.uint8).transpose(1, 2, 0)
Image.fromarray(img).save("sample.png")
print("saved sample.png")

/opt/anaconda3/envs/diffusion231/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


saved sample.png


In [40]:
import sys, transformers, diffusers, torch
print("python:", sys.executable)
print("transformers:", transformers.__version__, transformers.__file__)
print("diffusers:", diffusers.__version__)
print("torch:", torch.__version__)
from diffusers.utils import import_utils as iu
print("is_transformers_available:", iu.is_transformers_available())

python: /opt/anaconda3/envs/diffusion231/bin/python
transformers: 5.9.0 /opt/anaconda3/envs/diffusion231/lib/python3.12/site-packages/transformers/__init__.py
diffusers: 0.38.0
torch: 2.5.1
is_transformers_available: False


In [41]:
import sys

# Drop the stale 5.9.0 transformers + diffusers objects from memory
for mod in list(sys.modules):
    if mod.split(".")[0] in ("transformers", "diffusers"):
        del sys.modules[mod]

import transformers, diffusers
from diffusers.utils import import_utils as iu
print("transformers:", transformers.__version__)
print("diffusers:", diffusers.__version__)
print("is_transformers_available:", iu.is_transformers_available())

transformers: 4.57.6
diffusers: 0.38.0
is_transformers_available: True


In [42]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float32,
)
pipe = pipe.to("mps")
pipe.enable_attention_slicing()  # recommended on MPS to avoid memory spikes
print("Stable Diffusion 1.4 loaded:", type(pipe).__name__, "on", pipe.device)

Loading pipeline components...: 100%|██████████| 7/7 [00:00<00:00, 12.04it/s]


Stable Diffusion 1.4 loaded: StableDiffusionPipeline on mps:0
